## Identificacao

**Aluno:** *Sergio Leandro dos Santos*

---

**Dados:** Inside Airbnb — Rio de Janeiro, coleta 24/06/2026  
**Arquivo:** `listings.csv` (48.729 anúncios, 19 variáveis)

> ⚠️ Execute este notebook do início ao fim sem erros antes de submeter. Certifique-se de que `listings.csv` está na mesma pasta que este arquivo.

In [1]:
# Imports e configurações iniciais
import pandas as pd
import numpy as np
import altair as alt

# Altair: habilita renderização inline e remove limite de 5 000 linhas
alt.data_transformers.disable_max_rows()

# Carregamento dos dados
df = pd.read_csv('listings.csv')
print(f'Anúncios carregados: {len(df):,}')
print(f'Variáveis: {df.shape[1]}')
df.head()

Anúncios carregados: 48,729
Variáveis: 19


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,1025196153924029661,Quartos disponíveis no Recreio,546603190.0,1.470446e+18,Ananda,NaN,Recreio dos Bandeirantes,-23.016320,-43.462606,Private room,NaN,1.0,0,NaN,NaN,1.0,364,0,NaN
1,1697400829368754530,Casa próxima a Floresta da Tijuca,737983482.0,1.697376e+18,Yuri,NaN,Alto da Boa Vista,-22.975611,-43.284655,Private room,219.0,1.0,0,NaN,NaN,1.0,365,0,NaN
2,42874066,Alugo uma vaga para sexo feminino Recreio,341395950.0,1.469670e+18,Mabel,NaN,Recreio dos Bandeirantes,-23.022660,-43.492710,Private room,1659.0,30.0,0,NaN,NaN,1.0,269,0,NaN
3,1543530193994981165,Barra Vista maravilhosa,173836079.0,1.463060e+18,Roberta,NaN,Barra da Tijuca,-23.009319,-43.328026,Entire home/apt,704.0,3.0,0,NaN,NaN,19.0,5,0,NaN
4,831750365913119274,Suíte Rio BR,69238992.0,1.462790e+18,Júnior,NaN,Bento Ribeiro,-22.874314,-43.360450,Private room,575.0,21.0,0,NaN,NaN,1.0,83,0,NaN


In [2]:
# Exploracao inicial, qualidade dos dados e preparacao dos recortes
print(f'Dimensoes: {df.shape[0]:,} anuncios x {df.shape[1]} variaveis')
print(f'IDs duplicados: {df["id"].duplicated().sum():,}')
print('\nValores ausentes por variavel:')
print(df.isna().sum().sort_values(ascending=False).to_string())
print('\nTipos de acomodacao:')
print(df['room_type'].value_counts().to_string())
print('\nDez bairros com mais anuncios:')
print(df['neighbourhood'].value_counts().head(10).to_string())

# Precos positivos; o percentil 95 limita a cauda apenas nos graficos de distribuicao.
precos_validos = df.loc[df['price'].gt(0), 'price'].dropna()
limite_preco_p95 = precos_validos.quantile(0.95)
df_preco = df.loc[df['price'].gt(0) & df['price'].le(limite_preco_p95)].copy()
bairros_top10 = df['neighbourhood'].value_counts().head(10).index.tolist()
df_bairros_top10 = df.loc[df['neighbourhood'].isin(bairros_top10)].copy()

print(f'\nPreco: {len(precos_validos):,} valores positivos; mediana = R$ {precos_validos.median():,.2f}; '
      f'p95 = R$ {limite_preco_p95:,.2f}; maximo = R$ {precos_validos.max():,.2f}')
print(f'Anuncios sem preco: {df["price"].isna().sum():,}')
print(f'Anuncios no recorte de preco (ate p95): {len(df_preco):,}')
print('\nDisponibilidade (dias anunciados, nao ocupacao):')
print(df['availability_365'].describe().round(1).to_string())

Dimensoes: 48,729 anuncios x 19 variaveis
IDs duplicados: 0

Valores ausentes por variavel:
neighbourhood_group               48729
license                           48729
reviews_per_month                  9825
last_review                        9825
price                              4172
host_name                            17
host_profile_id                      15
host_id                              14
calculated_host_listings_count       14
minimum_nights                        7
id                                    0
name                                  0
longitude                             0
latitude                              0
neighbourhood                         0
room_type                             0
number_of_reviews                     0
availability_365                      0
number_of_reviews_ltm                 0

Tipos de acomodacao:
room_type
Entire home/apt    39471
Private room        8624
Shared room          601
Hotel room            33

Dez bairros com

## Parte 1 - Perguntas de Investigacao

Formule duas perguntas respondíveis com os dados.  
Para cada uma: justifique a relevância para o observatório e indique as variáveis (1–2 frases).

### Pergunta 1

**Pergunta:** As diferenças de preço anunciado entre bairros se mantêm quando comparamos o mesmo tipo de hospedagem, ou são explicadas em parte pela composição dos anúncios?

**Relevância e variáveis:** A comparação ajuda o Observatório a distinguir diferenças de preço dentro de cada tipo daquelas associadas à proporção de casas inteiras, quartos privados e compartilhados em cada bairro. Usarei `price`, `room_type` e `neighbourhood`, comparando medianas gerais por bairro com medianas por bairro e tipo nos dez bairros com mais anúncios.

### Pergunta 2

**Pergunta:** Como se distribuem o volume de anúncios e os dias de disponibilidade anunciada entre os principais bairros do Rio?

**Relevância e variáveis:** O retrato da oferta ativa e da disponibilidade declarada pode orientar onde aprofundar o monitoramento turístico, sem equivaler a reservas ou ocupação. Usarei `neighbourhood`, `room_type` e `availability_365`, restringindo a comparação aos dez bairros com mais anúncios para manter os gráficos legíveis.

## Parte 2 - Visualizações

Produza 3 a 4 gráficos que respondam às suas perguntas.  
Para **pelo menos dois**, justifique: (a) a escolha do tipo de gráfico e (b) as decisões de tratamento (filtragem, escala, exclusão de extremos).  
Todos os gráficos devem ter **título, eixos rotulados e legenda** quando houver mais de uma série.

### Gráfico 1 — Preço por bairro antes e depois de padronizar o tipo

**Justificativa do tipo:** O dumbbell mostra, para cada bairro, a média observada e a média padronizada, ligando os dois pontos. A distância entre eles evidencia quanto o preço muda ao aplicar a mesma composição de tipos a todos os bairros.

**Decisões de tratamento:** Excluí preços ausentes ou não positivos e limitei a análise ao p95 global (R$ 2.283). Considerei os dez bairros com mais anúncios e somente casas/apartamentos inteiros e quartos privados, pois são os únicos tipos com pelo menos 20 preços em cada bairro. A média observada usa a proporção local desses tipos; a padronizada usa pesos fixos comuns (85,5% e 14,5%, respectivamente). Uso médias para aplicar essa ponderação; o Gráfico 2 complementa com medianas por bairro e tipo. O eixo é marcado a cada R$25; os rótulos aparecem a cada R$50 para evitar sobreposição.

**Como interpretar as duas médias:** A média observada preserva a proporção local de casas/apartamentos e quartos privados em cada bairro. A média padronizada recalcula cada bairro com a mesma composição conjunta dos dez bairros: 85,5% casas/apartamentos e 14,5% quartos privados.

Por exemplo, Santa Teresa passa de R$ 419,27 observados para R$ 451,10 padronizados; em Leblon, passa de R$ 844,74 para R$ 820,32. A composição altera os níveis, mas não muda o ranking neste recorte; isso é uma comparação descritiva, não um efeito causal do bairro.


In [3]:
# Grafico 1: media observada versus media padronizada pela composicao de tipos
contagem_tipo_bairro = (
    df_preco.loc[df_preco['neighbourhood'].isin(bairros_top10)]
    .groupby(['neighbourhood', 'room_type'])['price']
    .size()
    .unstack(fill_value=0)
    .reindex(index=bairros_top10, fill_value=0)
)
# Mantém apenas tipos com suporte mínimo em todos os bairros.
tipos_comuns = [
    tipo for tipo in contagem_tipo_bairro.columns
    if contagem_tipo_bairro[tipo].ge(20).all()
]
if len(tipos_comuns) < 2:
    raise ValueError('Menos de dois tipos têm amostra suficiente em todos os bairros selecionados.')

# Usa preços válidos até o p95 e os mesmos tipos nos dez bairros.
dados_comparacao = df_preco.loc[
    df_preco['neighbourhood'].isin(bairros_top10)
    & df_preco['room_type'].isin(tipos_comuns)
].copy()
# Estes pesos comuns representam a composição conjunta dos dez bairros.
pesos_tipo = dados_comparacao['room_type'].value_counts(normalize=True).reindex(tipos_comuns)
medias_por_tipo = dados_comparacao.pivot_table(
    index='neighbourhood', columns='room_type', values='price', aggfunc='mean'
).reindex(index=bairros_top10)
# A média observada mantém a proporção local de cada tipo em cada bairro.
media_observada = dados_comparacao.groupby('neighbourhood')['price'].mean().reindex(bairros_top10)
# A padronizada aplica os mesmos pesos a todos os bairros.
media_padronizada = medias_por_tipo[tipos_comuns].mul(pesos_tipo, axis='columns').sum(axis=1)

comparacao_composicao = pd.DataFrame({
    'neighbourhood': bairros_top10,
    'media_observada': media_observada.to_numpy(),
    'media_padronizada': media_padronizada.to_numpy(),
    'anuncios_comparaveis': dados_comparacao.groupby('neighbourhood').size().reindex(bairros_top10).to_numpy()
})
ordem_bairros_preco = comparacao_composicao.sort_values('media_padronizada')['neighbourhood'].tolist()
comparacao_longa = comparacao_composicao.melt(
    id_vars=['neighbourhood', 'anuncios_comparaveis'],
    value_vars=['media_observada', 'media_padronizada'],
    var_name='medida', value_name='preco_medio'
)
comparacao_longa['medida'] = comparacao_longa['medida'].map({
    'media_observada': 'Média observada',
    'media_padronizada': 'Média padronizada'
})

# A escala mostra a faixa de R$300 a R$850, com divisões de R$25.
base_comparacao = alt.Chart(comparacao_longa).encode(
    x=alt.X(
        'preco_medio:Q', title='Preço médio aparado (R$)',
        scale=alt.Scale(domain=[300, 850]),
        axis=alt.Axis(format='$,.0f', values=list(range(300, 851, 25)), labelExpr="datum.value % 50 === 0 ? '$' + datum.value : ''")
    ),
    y=alt.Y(
        'neighbourhood:N', title='Bairro', sort=ordem_bairros_preco,
        scale=alt.Scale(padding=0.1)
    )
)
linhas_comparacao = base_comparacao.mark_line(color='#A9B3B1', strokeWidth=2).encode(
    detail='neighbourhood:N',
    order=alt.Order('medida:N', sort='ascending')
)
pontos_comparacao = base_comparacao.mark_point(filled=True, size=95).encode(
    color=alt.Color(
        'medida:N', title='Medida',
        scale=alt.Scale(
            domain=['Média observada', 'Média padronizada'],
            range=['#CC5A45', '#287C78']
        )
    ),
    tooltip=[
        alt.Tooltip('neighbourhood:N', title='Bairro'),
        alt.Tooltip('medida:N', title='Medida'),
        alt.Tooltip('preco_medio:Q', title='Preço médio (R$)', format=',.2f'),
        alt.Tooltip('anuncios_comparaveis:Q', title='Anúncios comparáveis', format=',')
    ]
)
grafico1 = (
    (linhas_comparacao + pontos_comparacao)
    .properties(
        title='Preço por bairro antes e depois de padronizar o tipo',
        width=650, height=300
    )
)
grafico1

alt.LayerChart(...)

### Gráfico 2 — Mediana do preço por bairro e tipo de acomodação

**Justificativa do tipo:** O mapa de calor compara simultaneamente bairro e tipo, e a cor representa a mediana do preço anunciado por noite. O tooltip mostra a mediana e o tamanho da amostra de cada combinação.

**Decisões de tratamento:** Uso os dez bairros com maior número de anúncios e os mesmos preços positivos até o p95 do Gráfico 1. Exibo apenas combinações com pelo menos 20 anúncios nesse recorte; células ausentes significam amostra insuficiente, não preço zero. A composição por tipo ainda pode explicar parte das diferenças entre bairros.
**Como interpretar:** Cada quadrado corresponde a um bairro e um tipo de hospedagem; a cor representa a mediana dos preços anunciados por noite, e o tooltip mostra o número de anúncios usados. Célula em branco indica menos de 20 preços válidos no grupo, não preço zero.

No recorte até o p95, por exemplo, casas/apartamentos inteiros têm mediana de R$ 628 na Barra da Tijuca e R$ 285 no Centro; para quartos privados, as medianas são R$ 373,50 e R$ 194. A comparação é descritiva e não controla atributos como tamanho ou comodidades.


In [4]:
# Grafico 2: mediana de preco por bairro e tipo, com n minimo por combinacao
# Resume cada bairro/tipo após os filtros de preço e bairro já aplicados.
resumo_preco = (
    df_preco.loc[df_preco['neighbourhood'].isin(bairros_top10)]
    .groupby(['neighbourhood', 'room_type'], as_index=False)
    .agg(mediana_preco=('price', 'median'), anuncios=('price', 'size'))
)
# Omite grupos com menos de 20 preços; ausência de célula não significa preço zero.
resumo_preco = resumo_preco.loc[resumo_preco['anuncios'].ge(20)].copy()

# A cor codifica a mediana; o tooltip informa o tamanho de cada grupo.
grafico2 = (
    alt.Chart(resumo_preco)
    .mark_rect(cornerRadius=2)
    .encode(
        x=alt.X('room_type:N', title='Tipo de acomodacao', sort=['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']),
        y=alt.Y('neighbourhood:N', title='Bairro', sort=bairros_top10),
        color=alt.Color('mediana_preco:Q', title='Mediana (R$)', scale=alt.Scale(scheme='tealblues')),
        tooltip=[
            alt.Tooltip('neighbourhood:N', title='Bairro'),
            alt.Tooltip('room_type:N', title='Tipo'),
            alt.Tooltip('mediana_preco:Q', title='Mediana (R$)', format=',.2f'),
            alt.Tooltip('anuncios:Q', title='Anuncios', format=',')
        ]
    )
    .properties(title='Mediana do preco por bairro e tipo (n >= 20)', width=650, height=360)
)
grafico2

alt.Chart(...)

### Gráfico 3 — Volume de anúncios por bairro e tipo de acomodação

In [5]:
# Grafico 3: volume de anuncios por bairro, dividido por tipo
contagem_bairro_tipo = (
    df_bairros_top10.groupby(['neighbourhood', 'room_type'], as_index=False)
    .size()
    .rename(columns={'size': 'anuncios'})
)
grafico3 = (
    alt.Chart(contagem_bairro_tipo)
    .mark_bar()
    .encode(
        x=alt.X('anuncios:Q', title='Numero de anuncios'),
        y=alt.Y('neighbourhood:N', title='Bairro', sort=bairros_top10),
        color=alt.Color('room_type:N', title='Tipo de acomodacao', scale=alt.Scale(scheme='tableau10')),
        tooltip=[alt.Tooltip('neighbourhood:N', title='Bairro'), alt.Tooltip('room_type:N', title='Tipo'), alt.Tooltip('anuncios:Q', title='Anuncios', format=',')]
    )
    .properties(title='Oferta ativa nos dez bairros com mais anuncios', width=650, height=360)
)
grafico3

alt.Chart(...)

### Gráfico 4 — Mediana de dias disponíveis por bairro

A disponibilidade é apresentada como dias declarados no calendário de 365 dias, não como ocupação observada. O gráfico usa a mediana e mostra o número de anúncios por bairro no tooltip; não excluí valores válidos do intervalo de 0 a 365.

In [6]:
# Grafico 4: mediana de dias anunciados disponiveis por bairro
resumo_disponibilidade = (
    df_bairros_top10.groupby('neighbourhood', as_index=False)
    .agg(mediana_dias=('availability_365', 'median'), anuncios=('id', 'size'))
    .sort_values('mediana_dias', ascending=True)
)
grafico4 = (
    alt.Chart(resumo_disponibilidade)
    .mark_bar(color='#287C78')
    .encode(
        x=alt.X('mediana_dias:Q', title='Mediana de dias disponíveis (0–365)', scale=alt.Scale(domain=[0, 365])),
        y=alt.Y('neighbourhood:N', title='Bairro', sort='-x'),
        tooltip=[alt.Tooltip('neighbourhood:N', title='Bairro'), alt.Tooltip('mediana_dias:Q', title='Mediana (dias)', format='.0f'), alt.Tooltip('anuncios:Q', title='Anuncios', format=',')]
    )
    .properties(title='Disponibilidade anunciada nos dez bairros com mais anuncios', width=650, height=360)
)
grafico4

alt.Chart(...)

### Dashboard — Capturas de Tela

O dashboard interativo está em `dashboard.py` e usa `listings.csv` na mesma pasta. Para executá-lo: `python -m streamlit run dashboard.py`.

![Dashboard — filtros padrão e comparação de preços](dashboard_screenshot_1.png)

![Dashboard — visão em janela ampla](dashboard_screenshot_2.png)

## Parte 3 - Conclusoes, Limitacoes e Reflexao Critica

Para cada pergunta: apresente a resposta suportada pelas evidências, pelo menos **duas limitações** dos dados que impedem generalizar, e uma investigação adicional que dados complementares permitiriam.  

Além disso, demonstre pensamento crítico sobre sua análise (ex.: alterar uma decisão de tratamento e verificar o efeito; apresentar explicação alternativa; verificar um número de forma independente).

### Conclusões — Pergunta 1

**Resposta:** Entre 44.557 anúncios com preço positivo, a mediana é R$ 453 e a média R$ 878. No recorte até o p95 (R$ 2.283), a mediana é R$ 466 para casas/apartamentos, R$ 296 para quartos privados e R$ 124 para quartos compartilhados. A diferença também aparece dentro dos bairros: casas/apartamentos têm mediana de R$ 628 na Barra da Tijuca e R$ 285 no Centro; quartos privados, R$ 373,50 e R$ 194, respectivamente. São preços anunciados, não valores efetivamente pagos.

**Limitações:**
1. 4.172 anúncios não têm preço; além disso, preço anunciado não inclui necessariamente taxas, descontos ou negociação e não mede receita.
2. O snapshot é de uma única data e não controla tamanho, qualidade, comodidades, sazonalidade ou distância de atrações; diferenças entre bairros não são efeito causal da localização.
3. O limite p95 oculta a cauda superior, e algumas combinações bairro/tipo são omitidas quando têm menos de 20 preços válidos. Há apenas 25 preços válidos de quartos de hotel no conjunto todo.

**Investigação adicional:** Cruzar com calendário histórico, reservas/diárias efetivamente pagas e características detalhadas dos imóveis permitiria separar sazonalidade, ocupação, taxas e diferenças de produto.

### Conclusões — Pergunta 2

**Resposta:** Os dez bairros com mais anúncios reúnem 37.312 registros (76,6% do conjunto); Copacabana sozinha tem 15.036. A oferta é majoritariamente de casas/apartamentos (81,0% dos anúncios). Entre os dez bairros, a mediana de disponibilidade declarada vai de 150 dias no Flamengo a 269 no Recreio dos Bandeirantes; Copacabana tem 170 dias. Isso descreve dias indicados como disponíveis, não noites ocupadas ou demanda.

**Limitações:**
1. `availability_365` é uma fotografia do calendário declarado; não permite distinguir datas bloqueadas pelo anfitrião de reservas nem calcular ocupação.
2. A coleta contém anúncios ativos em uma única data, não a oferta histórica, anúncios inativos ou acomodações fora da plataforma; os dez bairros selecionados cobrem 76,6%, não toda a oferta.
3. Contagens de anúncios não equivalem a unidades únicas ou capacidade de hóspedes; um anfitrião pode controlar vários anúncios e os imóveis têm capacidades diferentes.

**Investigação adicional:** Calendários diários ao longo de um ano, reservas confirmadas, capacidade dos imóveis e dados de hospedagem fora da plataforma permitiriam medir ocupação e comparar a oferta total por período e bairro.

### Reflexão Crítica

A média de preço (R$ 878) é quase o dobro da mediana (R$ 453), e o máximo chega a R$ 574.013; por isso, a média não representa bem o anúncio típico. O corte no p95 remove cerca de 4,9% dos preços positivos para tornar as distribuições legíveis. A ordem das medianas permanece igual nos dois cenários: hotel, casas/apartamentos, quartos privados e compartilhados. Porém, hotel tem apenas 25 preços válidos, então sua posição não deve ser generalizada; entre os três tipos mais numerosos, a ordem se mantém. O corte muda os níveis (por exemplo, a mediana de casas/apartamentos passa de R$ 489 para R$ 466), mas não demonstra que a localização causa preços diferentes nem garante que valores extremos sejam erros.

Como verificação independente, o código abaixo recalcula as medianas diretamente com pandas e confere a soma dos grupos do gráfico de volume contra o número de anúncios dos dez bairros.

In [7]:
# Sensibilidade ao corte de preco e verificacao independente das contagens
precos_positivos = df.loc[df['price'].gt(0), ['room_type', 'price']].dropna()
for rotulo, limite in [('sem corte', None), ('ate p95', limite_preco_p95)]:
    recorte = precos_positivos if limite is None else precos_positivos.loc[precos_positivos['price'].le(limite)]
    resumo = recorte.groupby('room_type')['price'].agg(['count', 'median', 'mean']).sort_values('median', ascending=False)
    print(f'\n{rotulo}:')
    print(resumo.round(1).to_string())

contagem_independente = df_bairros_top10.groupby(['neighbourhood', 'room_type']).size().sum()
assert df.shape == (48_729, 19)
assert df['id'].is_unique
assert contagem_independente == len(df_bairros_top10) == 37_312
print(f'\nVerificacao: {contagem_independente:,} contagens agrupadas = {len(df_bairros_top10):,} anuncios nos dez bairros.')


sem corte:
                 count  median   mean
room_type                            
Hotel room          25   582.0  600.0
Entire home/apt  36153   489.0  964.6
Private room      7795   300.0  521.9
Shared room        584   125.0  258.2

ate p95:
                 count  median   mean
room_type                            
Hotel room          25   582.0  600.0
Entire home/apt  34112   466.0  599.7
Private room      7674   296.0  405.6
Shared room        577   124.0  193.6

Verificacao: 37,312 contagens agrupadas = 37,312 anuncios nos dez bairros.


## Parte 4 - IA e Reprodutibilidade

Liste as ferramentas de IA usadas. Para um resultado relevante, descreva em até 5 linhas: o que foi pedido, o que foi produzido e o que você verificou, corrigiu ou descartou.

**Ferramentas de IA usadas:**
- GitHub Copilot no VS Code

**Exemplo de uso verificado:**

- **O que foi pedido:** Apoio para explorar o CSV, selecionar comparações e implementar gráficos e dashboard.
- **O que foi produzido:** Código de preparação, visualizações Altair, dashboard Streamlit e rascunho de interpretação.
- **O que foi verificado / corrigido / descartado:** Ao revisar o notebook, percebi que o Gráfico 1 ainda mostrava a faixa inteira de preços (mesmo os não praticados) e que as imagens do dashboard não carregavam; e se todas as perguntas da prova tinham sido respondidas.
